# Episode 17: A Customer Service System

Group 5 is about *applications* — combining the primitives into real systems. First up: an end-to-end customer-service triage system.

**In this episode you'll build:** a coordinator that classifies a support ticket and routes it to the right specialist.

## The design

A real support desk does two things: it **classifies** an incoming ticket, and it **routes** it to whoever can resolve it.

We build exactly that, reusing patterns you already know:

1. A **triage** agent with a `response_schema` (Episode 9) returns a typed classification.
2. A **coordinator** with three specialists wired in via `agent.as_tool()` (Episode 4) delegates the ticket.

This is tier-2 delegation — one coordinator drives. (For an audited, networked version you'd reach for the coordinator pattern of Episode 15.)

## Setup

In [ ]:
from dotenv import load_dotenv

load_dotenv()

from typing import Annotated

from pydantic import BaseModel, Field

from autogen.beta import Agent
from autogen.beta.config import OpenAIConfig

config = OpenAIConfig(model="gpt-4.1-mini", temperature=0)

ticket = "I was charged twice for my subscription this month. Can you help?"

## Step 1: Structured triage

The triage agent doesn't answer the ticket — it *classifies* it. A `response_schema` gives us a typed `Triage` object, so downstream code can branch on real fields instead of parsing text.

In [ ]:
class Triage(BaseModel):
    category: Annotated[str, Field(description="billing, technical, or general")]
    urgency: Annotated[str, Field(description="low, medium, or high")]


triage = Agent(
    "triage",
    prompt="Classify the support ticket.",
    config=config,
    response_schema=Triage,
)

classification = await triage.ask(ticket)
result = await classification.content()
print(f"triage -> category={result.category} urgency={result.urgency}")

## Step 2: A coordinator with specialists

Three specialist agents — billing, technical, general — each wrapped as a tool with `as_tool()`. The coordinator reads the ticket, picks the right specialist, and returns the resolution.

In [ ]:
billing = Agent(
    "billing",
    prompt="You are a billing specialist. Answer in 2 sentences.",
    config=config,
)
technical = Agent(
    "technical",
    prompt="You are a technical support specialist. Answer in 2 sentences.",
    config=config,
)
general = Agent(
    "general",
    prompt="You are a general support agent. Answer in 2 sentences.",
    config=config,
)

coordinator = Agent(
    "coordinator",
    prompt="You are a support coordinator. Read the ticket and delegate to exactly one "
    "specialist tool, then return their answer to the customer.",
    config=config,
    tools=[
        billing.as_tool(description="Handle billing and payment questions."),
        technical.as_tool(description="Handle technical and bug questions."),
        general.as_tool(description="Handle general questions."),
    ],
)

reply = await coordinator.ask(ticket)
print(reply.body)

## What you built

Two small, focused agents composed into a system:

- The **triage** agent is reusable on its own — its typed output could feed a dashboard, a priority queue, or an SLA timer.
- The **coordinator** never hard-codes routing. It reads the ticket and picks a specialist, exactly like the LLM picks a tool in Episode 3 — because to it, a specialist *is* a tool.

Each specialist has its own prompt and could have its own tools, model, or knowledge base. You can test and improve them in isolation.

## Additional content

**Triage feeding the coordinator.** Here triage and routing run independently. You could instead pass the `Triage` result into the coordinator's prompt, or use it to pick the model — urgent tickets get a stronger one.

**When to go networked.** This tier-2 design is simple and fast. Move to a `workflow` channel (Episode 15's coordinator pattern) when you need an audit trail, when specialists are independently owned, or when routing itself becomes multi-step.

**Specialists with tools.** A real billing agent would have a `lookup_invoice` tool; a technical agent a `search_known_issues` tool. The structure doesn't change — you just fill each specialist's `tools=` slot.

## What's next

This system answered one ticket in one pass. Episode 18 adds a loop — agents that critique and revise each other's work until it's good enough.